In [9]:
from data_frame.columns.expression_builder import ExpressionBuilder
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, date
from data_frame.spark_utils import get_spark

In [10]:
spark = get_spark(app_name="Column Expressions")

In [11]:
data = [("Alice", 25, 50000), ("Bob", 35, 80000), ("Charlie", 45, 60000)]
df = spark.createDataFrame(data, ["name", "age", "salary"])

In [12]:
# Create conditional column
df = df.withColumn(
    "salary_category",
    ExpressionBuilder.conditional_column(
        F.col("salary") > 70000, "High",
        ExpressionBuilder.conditional_column(
            F.col("salary") > 55000, "Medium", "Low"
        )
    )
)

df.show()

+-------+---+------+---------------+
|   name|age|salary|salary_category|
+-------+---+------+---------------+
|  Alice| 25| 50000|            Low|
|    Bob| 35| 80000|           High|
|Charlie| 45| 60000|         Medium|
+-------+---+------+---------------+



In [14]:
"""
## 2. Date Operations
"""

date_data = [(1, datetime.now()), (2, date(2023, 1, 15))]
df_dates = spark.createDataFrame(date_data, ["id", "event_date"])

# Derive date components
df_dates = ExpressionBuilder.derive_from_date(
    df_dates, "event_date", 
    ["year", "month", "day", "day_of_week", "quarter"]
)
df_dates.show()

PySparkTypeError: [CANNOT_MERGE_TYPE] Can not merge type `TimestampType` and `DateType`.

In [15]:
"""
## 3. String Operations
"""
string_data = [("  Hello World  ", "Spark"), ("  PySpark  ", "DataFrame")]
df_strings = spark.createDataFrame(string_data, ["text1", "text2"])

# Apply string operations
df_strings = ExpressionBuilder.string_operations(
    df_strings, "text1",
    {
        "upper": ["text1", "text2"],
        "length": "text1",
        "trim": "text1"
    }
)
df_strings.show(truncate=False)

+---------------+---------+---------------+-----------+------------+-----------+
|text1          |text2    |text1_upper    |text2_upper|text1_length|text1_trim |
+---------------+---------+---------------+-----------+------------+-----------+
|  Hello World  |Spark    |  HELLO WORLD  |SPARK      |15          |Hello World|
|  PySpark      |DataFrame|  PYSPARK      |DATAFRAME  |11          |PySpark    |
+---------------+---------+---------------+-----------+------------+-----------+

